# India Air Pollution Analytics

## Objective

This project analyzes pollutant observations collected from monitoring stations across Indian cities and states. It identifies major pollutants, high-observed-pollution locations, state-wise differences, monthly patterns, and potential outliers.

## Dataset Note

The dataset contains pollutant measurements but does not contain a direct official AQI column. Therefore, this project analyzes observed air-pollution patterns and does not calculate official AQI.

In [1]:
# ============================================================
# TASK 2: EXPLORATORY DATA ANALYSIS
# INDIAN AIR POLLUTION DATASET
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", context="notebook")

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

file_name = "3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Inspect dataset structure
# ------------------------------------------------------------

print("COLUMN NAMES")
print(df.columns.tolist())

print("DATA TYPES")
print(df.dtypes)

print("FIRST FIVE ROWS")
display(df.head())

print("LAST FIVE ROWS")
display(df.tail())

print("DATASET INFORMATION")
df.info()

# ------------------------------------------------------------
# 3. Data quality checks
# ------------------------------------------------------------

print("MISSING VALUES BEFORE CLEANING")

missing_before = df.isnull().sum().to_frame("Missing Values")
display(missing_before)

print("DUPLICATE ROWS BEFORE CLEANING")
duplicates_before = df.duplicated().sum()
print(duplicates_before)

# ------------------------------------------------------------
# 4. Data cleaning
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

df["last_update"] = pd.to_datetime(
    df["last_update"],
    dayfirst=True,
    errors="coerce"
)

numeric_columns = [
    "latitude",
    "longitude",
    "pollutant_min",
    "pollutant_max",
    "pollutant_avg"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

rows_before_duplicates = len(df)

df = df.drop_duplicates().copy()

rows_after_duplicates = len(df)

print("DUPLICATE REMOVAL")
print("Rows before removing duplicates:", rows_before_duplicates)
print("Rows after removing duplicates:", rows_after_duplicates)
print("Duplicates removed:", rows_before_duplicates - rows_after_duplicates)

# Fill numeric missing values using median
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Fill text missing values using mode, if any exist
categorical_columns = [
    column for column in df.columns
    if df[column].dtype == "object"
]

for column in categorical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])

# Create time features
df["month"] = df["last_update"].dt.to_period("M").astype(str)
df["year"] = df["last_update"].dt.year
df["month_number"] = df["last_update"].dt.month

print("DATA CLEANING COMPLETED")
print("Final shape:", df.shape)

print("MISSING VALUES AFTER CLEANING")
missing_after = df.isnull().sum().to_frame("Missing Values")
display(missing_after)

# ------------------------------------------------------------
# 5. Dataset overview
# ------------------------------------------------------------

overview = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Total countries",
        "Total states",
        "Total cities",
        "Total stations",
        "Total pollutants",
        "Starting date",
        "Ending date",
        "Duplicate rows remaining",
        "Missing values remaining"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["country"].nunique(),
        df["state"].nunique(),
        df["city"].nunique(),
        df["station"].nunique(),
        df["pollutant_id"].nunique(),
        df["last_update"].min(),
        df["last_update"].max(),
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

print("DATASET OVERVIEW")
display(overview)

# ------------------------------------------------------------
# 6. Descriptive statistics
# ------------------------------------------------------------

print("DESCRIPTIVE STATISTICS")

descriptive_statistics = df[
    [
        "latitude",
        "longitude",
        "pollutant_min",
        "pollutant_max",
        "pollutant_avg"
    ]
].describe().round(2)

display(descriptive_statistics)

print("MEDIAN VALUES")

median_values = df[
    [
        "pollutant_min",
        "pollutant_max",
        "pollutant_avg"
    ]
].median().round(2)

display(median_values.to_frame("Median"))

print("SKEWNESS VALUES")

skewness_values = df[
    [
        "pollutant_min",
        "pollutant_max",
        "pollutant_avg"
    ]
].skew().round(2)

display(skewness_values.to_frame("Skewness"))

# ------------------------------------------------------------
# 7. Categorical analysis
# ------------------------------------------------------------

print("POLLUTANTS AVAILABLE")
print(df["pollutant_id"].unique())

print("RECORDS BY POLLUTANT")
display(
    df["pollutant_id"]
    .value_counts()
    .to_frame("Records")
)

print("TOP 10 STATES BY NUMBER OF RECORDS")
display(
    df["state"]
    .value_counts()
    .head(10)
    .to_frame("Records")
)

print("TOP 10 CITIES BY NUMBER OF RECORDS")
display(
    df["city"]
    .value_counts()
    .head(10)
    .to_frame("Records")
)

# ------------------------------------------------------------
# 8. Pollutant-level analysis
# ------------------------------------------------------------

pollutant_summary = (
    df.groupby("pollutant_id")
      .agg(
          average_value=("pollutant_avg", "mean"),
          median_value=("pollutant_avg", "median"),
          minimum_value=("pollutant_min", "min"),
          maximum_value=("pollutant_max", "max"),
          standard_deviation=("pollutant_avg", "std"),
          records=("pollutant_avg", "count")
      )
      .round(2)
      .sort_values("average_value", ascending=False)
)

print("POLLUTANT SUMMARY")
display(pollutant_summary)

# ------------------------------------------------------------
# 9. City-level analysis
# ------------------------------------------------------------

city_summary = (
    df.groupby("city")
      .agg(
          average_pollution=("pollutant_avg", "mean"),
          median_pollution=("pollutant_avg", "median"),
          maximum_pollution=("pollutant_max", "max"),
          stations=("station", "nunique"),
          pollutants_monitored=("pollutant_id", "nunique"),
          records=("pollutant_avg", "count")
      )
      .round(2)
      .sort_values("average_pollution", ascending=False)
)

print("TOP 15 CITIES BY AVERAGE OBSERVED POLLUTION")
display(city_summary.head(15))

# ------------------------------------------------------------
# 10. State-level analysis
# ------------------------------------------------------------

state_summary = (
    df.groupby("state")
      .agg(
          average_pollution=("pollutant_avg", "mean"),
          median_pollution=("pollutant_avg", "median"),
          maximum_pollution=("pollutant_max", "max"),
          cities=("city", "nunique"),
          stations=("station", "nunique"),
          records=("pollutant_avg", "count")
      )
      .round(2)
      .sort_values("average_pollution", ascending=False)
)

print("TOP 15 STATES BY AVERAGE OBSERVED POLLUTION")
display(state_summary.head(15))

# ------------------------------------------------------------
# 11. Highest pollution observations
# ------------------------------------------------------------

print("TOP 20 HIGHEST POLLUTION OBSERVATIONS")

highest_records = (
    df.sort_values("pollutant_avg", ascending=False)
      .head(20)
)

display(
    highest_records[
        [
            "country",
            "state",
            "city",
            "station",
            "pollutant_id",
            "pollutant_min",
            "pollutant_max",
            "pollutant_avg",
            "last_update"
        ]
    ]
)

# ------------------------------------------------------------
# 12. IQR-based outlier detection
# ------------------------------------------------------------

def find_iqr_outliers(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outliers = dataframe[
        (dataframe[column] < lower_limit) |
        (dataframe[column] > upper_limit)
    ]

    return outliers, lower_limit, upper_limit

outliers, lower_limit, upper_limit = find_iqr_outliers(
    df,
    "pollutant_avg"
)

print("OUTLIER ANALYSIS USING IQR METHOD")
print("Q1 lower limit:", round(lower_limit, 2))
print("Upper limit:", round(upper_limit, 2))
print("Number of potential outliers:", len(outliers))
print("Percentage of potential outliers:",
      round(len(outliers) / len(df) * 100, 2), "%")

print("SAMPLE POTENTIAL OUTLIERS")

display(
    outliers.sort_values(
        "pollutant_avg",
        ascending=False
    )[
        [
            "state",
            "city",
            "station",
            "pollutant_id",
            "pollutant_avg",
            "last_update"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 13. Monthly analysis
# ------------------------------------------------------------

monthly_summary = (
    df.groupby("month")
      .agg(
          average_pollution=("pollutant_avg", "mean"),
          median_pollution=("pollutant_avg", "median"),
          maximum_pollution=("pollutant_max", "max"),
          records=("pollutant_avg", "count")
      )
      .round(2)
)

print("MONTHLY POLLUTION SUMMARY")
display(monthly_summary)

# ------------------------------------------------------------
# 14. Correlation analysis
# ------------------------------------------------------------

correlation_columns = [
    "pollutant_min",
    "pollutant_max",
    "pollutant_avg"
]

correlation_matrix = df[correlation_columns].corr().round(3)

print("CORRELATION MATRIX")
display(correlation_matrix)

# ------------------------------------------------------------
# 15. Statistical hypothesis test
# ------------------------------------------------------------

print("HYPOTHESIS TEST")

print("H0: There is no monotonic relationship between pollutant_min and pollutant_avg.")
print("H1: There is a monotonic relationship between pollutant_min and pollutant_avg.")

correlation_value, p_value = spearmanr(
    df["pollutant_min"],
    df["pollutant_avg"]
)

print("Spearman correlation:",
      round(correlation_value, 3))

print("P-value:",
      round(p_value, 5))

if p_value < 0.05:
    print("Decision: Reject H0.")
    print("Interpretation: The relationship is statistically significant.")
else:
    print("Decision: Fail to reject H0.")
    print("Interpretation: The relationship is not statistically significant.")

# ------------------------------------------------------------
# 16. Data issues and limitations report
# ------------------------------------------------------------

data_issues = pd.DataFrame({
    "Issue": [
        "Direct AQI column",
        "Missing values",
        "Duplicate records",
        "Potential outliers",
        "Different pollutant scales",
        "Station coverage"
    ],
    "Observation": [
        "No direct official AQI column is available.",
        "Missing values were checked and handled for analysis.",
        "Exact duplicate rows were removed.",
        "IQR method identified potential extreme observations.",
        "Pollutants may use different units and should not be compared directly without context.",
        "Cities with more monitoring stations may have more observations."
    ],
    "Recommended Action": [
        "Do not call this official AQI analysis.",
        "Verify missing-value treatment before advanced modelling.",
        "Keep a record of removed duplicates.",
        "Investigate whether extreme values are errors or genuine events.",
        "Use pollutant-specific standards or verified AQI breakpoints.",
        "Interpret city rankings as observed patterns, not complete health-risk rankings."
    ]
})

print("DATA ISSUES AND RECOMMENDATIONS")
display(data_issues)

# ------------------------------------------------------------
# 17. Save EDA outputs
# ------------------------------------------------------------

os.makedirs("eda_outputs", exist_ok=True)

overview.to_csv(
    "eda_outputs/dataset_overview.csv",
    index=False
)

missing_before.to_csv(
    "eda_outputs/missing_values_before_cleaning.csv"
)

missing_after.to_csv(
    "eda_outputs/missing_values_after_cleaning.csv"
)

descriptive_statistics.to_csv(
    "eda_outputs/descriptive_statistics.csv"
)

pollutant_summary.to_csv(
    "eda_outputs/pollutant_summary.csv"
)

city_summary.to_csv(
    "eda_outputs/city_summary.csv"
)

state_summary.to_csv(
    "eda_outputs/state_summary.csv"
)

monthly_summary.to_csv(
    "eda_outputs/monthly_summary.csv"
)

highest_records.to_csv(
    "eda_outputs/highest_pollution_records.csv",
    index=False
)

outliers.to_csv(
    "eda_outputs/potential_outliers.csv",
    index=False
)

correlation_matrix.to_csv(
    "eda_outputs/correlation_matrix.csv"
)

data_issues.to_csv(
    "eda_outputs/data_issues_and_recommendations.csv",
    index=False
)

df.to_csv(
    "eda_outputs/cleaned_air_pollution_data.csv",
    index=False
)

print("" + "=" * 60)
print("TASK 2 EDA COMPLETED SUCCESSFULLY")
print("=" * 60)
print("EDA tables and cleaned data are saved in the eda_outputs folder.")

Dataset loaded successfully!
Original shape: (3500, 11)
COLUMN NAMES
['country', 'state', 'city', 'station', 'last_update', 'latitude', 'longitude', 'pollutant_id', 'pollutant_min', 'pollutant_max', 'pollutant_avg']
DATA TYPES
country              str
state                str
city                 str
station              str
last_update          str
latitude         float64
longitude        float64
pollutant_id         str
pollutant_min    float64
pollutant_max    float64
pollutant_avg    float64
dtype: object
FIRST FIVE ROWS


,country,state,city,station,last_update,latitude,longitude,pollutant_id,pollutant_min,pollutant_max,pollutant_avg
0,India,Bihar,Gaya,"Kareemganj, Gaya - BSPCB",23-08-2026 11:00:00,24.792403,84.992416,PM2.5,NaN,NaN,NaN
1,India,Bihar,Hajipur,"Industrial Area, Hajipur - BSPCB",23-08-2026 11:00:00,25.697189,85.245900,CO,19.0,33.0,27.0
2,India,Bihar,Katihar,"Mirchaibari, Katihar - BSPCB",23-08-2026 11:00:00,25.560083,87.553265,OZONE,19.0,38.0,28.0
3,India,Bihar,Kishanganj,"SDM Office_Khagra, Kishanganj - BSPCB",23-08-2026 11:00:00,26.088130,87.938403,PM10,31.0,60.0,44.0
4,India,Bihar,Kishanganj,"SDM Office_Khagra, Kishanganj - BSPCB",23-08-2026 11:00:00,26.088130,87.938403,NO2,7.0,8.0,7.0


LAST FIVE ROWS


,country,state,city,station,last_update,latitude,longitude,pollutant_id,pollutant_min,pollutant_max,pollutant_avg
3495,India,West Bengal,Kolkata,"Fort William, Kolkata - WBPCB",23-08-2026 11:00:00,22.556640,88.342674,OZONE,30.0,40.0,36.0
3496,India,West Bengal,Kolkata,"Rabindra Sarobar, Kolkata - WBPCB",23-08-2026 11:00:00,22.511060,88.351420,NH3,6.0,6.0,6.0
3497,India,West Bengal,Kolkata,"Ballygunge, Kolkata - WBPCB",23-08-2026 11:00:00,22.536751,88.363802,NH3,7.0,8.0,8.0
3498,India,West Bengal,Kolkata,"Ballygunge, Kolkata - WBPCB",23-08-2026 11:00:00,22.536751,88.363802,OZONE,12.0,18.0,16.0
3499,India,West Bengal,Siliguri,"Ward-32 Bapupara, Siliguri - WBPCB",23-08-2026 11:00:00,26.687923,88.415250,NO2,112.0,130.0,120.0


DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        3500 non-null   str    
 1   state          3500 non-null   str    
 2   city           3500 non-null   str    
 3   station        3500 non-null   str    
 4   last_update    3500 non-null   str    
 5   latitude       3500 non-null   float64
 6   longitude      3500 non-null   float64
 7   pollutant_id   3500 non-null   str    
 8   pollutant_min  3217 non-null   float64
 9   pollutant_max  3217 non-null   float64
 10  pollutant_avg  3217 non-null   float64
dtypes: float64(5), str(6)
memory usage: 557.1 KB
MISSING VALUES BEFORE CLEANING


...

CORRELATION MATRIX


,pollutant_min,pollutant_max,pollutant_avg
pollutant_min,1.000,0.533,0.812
pollutant_max,0.533,1.000,0.878
pollutant_avg,0.812,0.878,1.000


HYPOTHESIS TEST
H0: There is no monotonic relationship between pollutant_min and pollutant_avg.
H1: There is a monotonic relationship between pollutant_min and pollutant_avg.
Spearman correlation: 0.889
P-value: 0.0
Decision: Reject H0.
Interpretation: The relationship is statistically significant.
DATA ISSUES AND RECOMMENDATIONS


,Issue,Observation,Recommended Action
0,Direct AQI column,No direct official AQI column is available.,Do not call this official AQI analysis.
1,Missing values,Missing values were checked and handled for an...,Verify missing-value treatment before advanced...
2,Duplicate records,Exact duplicate rows were removed.,Keep a record of removed duplicates.
3,Potential outliers,IQR method identified potential extreme observ...,Investigate whether extreme values are errors ...
4,Different pollutant scales,Pollutants may use different units and should ...,Use pollutant-specific standards or verified A...
5,Station coverage,Cities with more monitoring stations may have ...,"Interpret city rankings as observed patterns, ..."


TASK 2 EDA COMPLETED SUCCESSFULLY
EDA tables and cleaned data are saved in the eda_outputs folder.


## 5. EDA Findings

The dataset was examined for structure, data types, missing values, duplicate records, statistical characteristics, geographic coverage, pollutant-level differences, temporal trends, and potential outliers.

The analysis provides comparisons across pollutants, cities, states, stations, and months. The pollutant summary identifies pollutants with higher observed average values, while city and state summaries identify locations with higher observed pollution measurements.

## 6. Data Quality Findings

Duplicate records were checked and removed before analysis. Numeric columns were converted into numeric format, and missing numeric values were handled using median imputation.

Potential outliers were identified using the Interquartile Range method. These values should be investigated further because they may represent genuine pollution events, station-level variation, or measurement issues.

## 7. Hypothesis-Test Interpretation

The Spearman correlation test examined the relationship between `pollutant_min` and `pollutant_avg`.

The p-value printed by the code determines the result:

- If the p-value is below 0.05, the relationship is statistically significant.
- If the p-value is 0.05 or above, the relationship is not statistically significant.

Statistical significance indicates evidence of a relationship in the dataset; it does not prove that one variable causes the other.

## 8. Limitations

- The dataset does not contain a direct official AQI column.
- Different pollutants may have different units and measurement scales.
- Cities with more monitoring stations may have more records.
- Potential outliers were identified but not automatically deleted.
- The results describe observed pollution patterns and should not be interpreted as complete health-risk rankings.

## 9. Conclusion

This exploratory analysis established the structure, quality, and major characteristics of the Indian air-pollution dataset. It identified pollutant differences, high-observed-pollution cities and states, monthly patterns, potential outliers, and relationships among pollution measurements.

The cleaned data and summary tables can be used for Task 3, where the findings will be converted into professional visualizations and data stories.